# NGS Library Normalization & Liquid-Handling QC Exploration

This notebook demonstrates the computational quality-control (QC) and transfer-volume planning workflow for normalizing next-generation sequencing (NGS) libraries prior to equimolar or mass-balanced pooling.

## Workflow Objectives
1. Inspect raw fluorometric quantification (`conc_ng_ul`) and available eluate volume (`volume_ul`) measurements across a 24-library batch.
2. Apply the $C_1 \times V_1 = C_2 \times V_2$ dilution model for a target concentration of **10.0 ng/µL** in a final volume of **20.0 µL** (total mass = **200 ng**).
3. Evaluate automated liquid-handling QC constraints (stock concentration floor, available volume sufficiency, minimum pipetting volume $\ge 1.0$ µL, and rounding tolerance $\le 0.5\%$)."

In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [2]:
df = pd.read_csv("../results/normalized_samples.csv")
summary = json.loads(Path("../results/qc_summary.json").read_text())
print(f"Total libraries: {summary['total']} | PASS: {summary['passed']} ({summary['passed']/summary['total']*100:.1f}%) | FAIL: {summary['failed']} ({summary['failed']/summary['total']*100:.1f}%)")
print("Failure breakdown:", summary["failure_reasons"])
df.head(10)

Total libraries: 24 | PASS: 16 (66.7%) | FAIL: 8 (33.3%)
Failure breakdown: {'insufficient_volume': 2, 'stock_below_target': 6}


In [3]:
# Inspect libraries flagged as FAIL during pre-pooling normalization QC
df[df["status"] == "FAIL"][[
    "sample_id", "conc_ng_ul", "volume_ul", "required_sample_volume_ul", "status", "reason"
]]

In [4]:
# Summary statistics for actionable (PASS) pipetting volumes
pass_df = df[df["status"] == "PASS"].copy()
pass_df[["conc_ng_ul", "sample_volume_ul", "buffer_volume_ul", "predicted_conc_ng_ul"]].describe().round(2)